# `Cleaner` - Nettoyage des données agricoles

**Module :** `kadi.kidas.cleaner`  
**Classe :** `Cleaner`

---

## Introduction

`Cleaner` est la classe de nettoyage des données agricoles tabulaires. Elle offre :

- **Suppression des doublons** : exacts ou partiels (sous-ensembles de colonnes)
- **Imputation des NaN** : 4 stratégies - `mean`, `median`, `forward_fill`, `drop`
- **Détection des outliers** : 3 méthodes - IQR (Tukey), Z-score, MAD
- **Correction des dates** : normalisation des formats hétérogènes en `datetime64`
- **Standardisation du texte** : trim, suppression des accents, casse
- **Rapport détaillé** : journal de toutes les opérations effectuées

L'API est **chainable** : chaque méthode retourne le DataFrame modifié.

## 1. Création des données de démonstration

In [ ]:
import os
import pandas as pd

import numpy as np
import pandas as pd

from kadi.kidas.cleaner import Cleaner

# DataFrame avec tous les problèmes courants en AgriTech
df_brut = pd.DataFrame({
    'culture':         ['Maïs', 'Niébé', 'igname', 'MAÏS', None, 'sorgho', 'manioc', 'Maïs'],
    'commune':         ['Cotonou', 'Parakou', 'Bohicon', 'Cotonou', 'Kandi', 'Natitingou', 'Abomey', 'Cotonou'],
    'rendement_kg':    [1500, 800, 2000, 1500, np.nan, 1200, 3500, 1500],
    'prix_xof':        [350, 500, 250, 350, 300, 280, 180, 350],
    'outlier_col':     [1200, 1300, 1250, 1280, 15000, 1190, 1310, 1200],  # 15000 = outlier
    'date_recolte':    ['2024-01-15', '2024-02-20', '2024/03/10', '15-01-2024', None, '2024-05-01', '2024-06-15', '2024-01-15'],
    'notes':           ['  OK  ', 'Bon rendement!', 'C.A.S normal', 'Données OK', None, 'Bonne qualité', 'Stock épuisé', '  OK  '],
})

print(f"DataFrame brut : {len(df_brut)} lignes × {len(df_brut.columns)} colonnes")
df_brut

## 2. Initialisation de `Cleaner`

In [ ]:
# Le constructeur crée une copie interne du DataFrame
cleaner = Cleaner(df_brut)
print(f"Cleaner initialisé avec {cleaner.report()['lignes_initiales']} lignes")

## 3. `drop_dupes()` - Suppression des doublons

In [ ]:
cleaner = Cleaner(df_brut)

# Suppression des doublons exacts (toutes colonnes)
df_sans_dup = cleaner.drop_dupes()
print(f"Lignes avant : {len(df_brut)} → après : {len(df_sans_dup)}")
df_sans_dup

In [ ]:
# Suppression basée sur un sous-ensemble de colonnes
cleaner2 = Cleaner(df_brut)
df_dup_partiel = cleaner2.drop_dupes(subset=['culture', 'commune'], keep='first')
print(f"Doublons (culture+commune) supprimés : {len(df_brut) - len(df_dup_partiel)}")

## 4. `fill_missing()` - Traitement des valeurs manquantes

**Stratégies disponibles :** `'mean'`, `'median'`, `'forward_fill'`, `'drop'`

In [ ]:
print(f"NaN dans 'rendement_kg' avant traitement : {df_brut['rendement_kg'].isna().sum()}")

# Stratégie 1 : imputation par la moyenne
cleaner = Cleaner(df_brut)
df_mean = cleaner.fill_missing(strategy='mean', cols=['rendement_kg'])
print(f"Après 'mean'  : NaN restants = {df_mean['rendement_kg'].isna().sum()}")
print(f"Valeur imputée (moy) : {df_mean['rendement_kg'].iloc[4]:.1f} kg")

In [ ]:
# Stratégie 2 : imputation par la médiane
cleaner = Cleaner(df_brut)
df_median = cleaner.fill_missing(strategy='median', cols=['rendement_kg'])
print(f"Valeur imputée (médiane) : {df_median['rendement_kg'].iloc[4]:.1f} kg")

In [ ]:
# Stratégie 3 : forward fill (propage la valeur précédente)
cleaner = Cleaner(df_brut)
df_ffill = cleaner.fill_missing(strategy='forward_fill')
print(f"NaN restants (ffill) : {df_ffill.isna().sum().sum()}")

In [ ]:
# Stratégie 4 : suppression des lignes avec NaN
cleaner = Cleaner(df_brut)
df_drop = cleaner.fill_missing(strategy='drop')
print(f"Lignes supprimées : {len(df_brut) - len(df_drop)}")

## 5. `drop_outliers()` - Détection et suppression des outliers

**Méthodes disponibles :** `'iqr'` (Tukey), `'zscore'`, `'mad'` (robuste)

In [ ]:
# Méthode IQR (règle de Tukey : threshold=1.5 standard)
cleaner = Cleaner(df_brut)
df_propre, df_outliers = cleaner.drop_outliers(method='iqr', threshold=1.5, cols=['outlier_col'])

print(f"Outliers détectés (IQR) : {len(df_outliers)}")
print(f"Valeur identifiée comme outlier : {df_outliers['outlier_col'].values}")

In [ ]:
# Méthode Z-score (seuil = 3 déviations standard)
cleaner = Cleaner(df_brut)
df_propre_z, df_out_z = cleaner.drop_outliers(method='zscore', threshold=2.5, cols=['outlier_col'])
print(f"Outliers (Z-score, threshold=2.5) : {len(df_out_z)}")

In [ ]:
# Méthode MAD (Median Absolute Deviation, robuste aux extrêmes)
cleaner = Cleaner(df_brut)
df_propre_mad, df_out_mad = cleaner.drop_outliers(method='mad', threshold=3.0, cols=['outlier_col'])
print(f"Outliers (MAD, threshold=3.0) : {len(df_out_mad)}")

## 6. `parse_dates()` - Normalisation des formats de dates

In [ ]:
print("Formats de dates brutes :")
print(df_brut['date_recolte'].dropna().tolist())

cleaner = Cleaner(df_brut)
df_dates = cleaner.parse_dates(cols=['date_recolte'])

print(f"\nType après normalisation : {df_dates['date_recolte'].dtype}")
print("Valeurs converties :")
df_dates[['date_recolte', 'culture']].dropna()

## 7. `norm_text()` - Standardisation du texte

In [ ]:
print("Textes bruts dans 'culture' :")
print(df_brut['culture'].tolist())

# Conversion en minuscules + suppression des espaces et accents
cleaner = Cleaner(df_brut)
df_text = cleaner.norm_text(cols=['culture', 'commune'], case='lower')

print("\nAprès standardisation (lower) :")
print(df_text['culture'].tolist())

In [ ]:
# Titre (chaque mot commence par une majuscule)
cleaner = Cleaner(df_brut)
df_title = cleaner.norm_text(cols=['commune'], case='title')
print("Communes en titre :")
print(df_title['commune'].tolist())

## 8. `strip_chars()` - Suppression des caractères spéciaux

In [ ]:
print("Notes brutes :")
print(df_brut['notes'].tolist())

cleaner = Cleaner(df_brut)
df_notes = cleaner.strip_chars(cols=['notes'], keep='-')

print("\nNotes nettoyées :")
print(df_notes['notes'].tolist())

## 9. `check_decimals()` - Détection des séparateurs décimaux mixtes

In [ ]:
# Données avec mélange de '.' et ',' comme séparateur décimal
df_decimales = pd.DataFrame({
    'prix_str': ['1.500', '2,300', '1.800', '3,150', '2.000'],
    'poids_str': ['50.5', '48.0', '51,3', '49.8', '50.1'],
})

cleaner = Cleaner(df_decimales)
rapport_dec = cleaner.check_decimals(cols=['prix_str', 'poids_str'])

print("Rapport des séparateurs décimaux :")
for col, info in rapport_dec.items():
    print(f"\n  Colonne '{col}' :")
    for cle, val in info.items():
        print(f"    {cle:15s} : {val}")

## 10. Pipeline chainé - Exemple complet AgriTech

In [ ]:
# Pipeline de nettoyage complet enchaîné
cleaner = Cleaner(df_brut)

df_propre = cleaner.drop_dupes()
df_propre = cleaner.fill_missing(strategy='mean', cols=['rendement_kg', 'prix_xof'])
df_propre = cleaner.norm_text(cols=['culture', 'commune'], case='lower')
df_propre = cleaner.parse_dates(cols=['date_recolte'])
df_propre = cleaner.strip_chars(cols=['notes'])
df_propre, _ = cleaner.drop_outliers(method='iqr', cols=['outlier_col'])

print(f"DataFrame nettoyé : {len(df_propre)} lignes × {len(df_propre.columns)} colonnes")
df_propre

## 11. `report()` - Rapport des opérations effectuées

In [ ]:
rapport = cleaner.report()

print("=== Rapport de Nettoyage ===")
print(f"Lignes initiales       : {rapport['lignes_initiales']}")
print(f"Lignes finales         : {rapport['lignes_finales']}")
print(f"Doublons supprimés     : {rapport['doublons_supprimes']}")
print(f"NaN traités            : {rapport['nan_traites']}")
print(f"Outliers supprimés     : {rapport['outliers_detectes']}")
print(f"Dates corrigées        : {rapport['dates_corrigees']}")
print(f"\nOpérations effectuées ({len(rapport['operations'])}) :")
for op in rapport['operations']:
    print(f"  - {op['operation']}")

## Récapitulatif des méthodes

| Méthode | Description |
|---|---|
| `drop_dupes(subset, keep)` | Supprime les lignes dupliquées |
| `fill_missing(strategy, columns)` | Impute les NaN (mean/median/ffill/drop) |
| `drop_outliers(method, threshold, columns)` | Détecte les outliers (IQR/Z-score/MAD) |
| `parse_dates(columns)` | Normalise les formats de dates en datetime64 |
| `norm_text(columns, case)` | Trim, accents, casse |
| `strip_chars(cols, keep)` | Supprime les caractères spéciaux |
| `check_decimals(columns)` | Détecte les mélanges `.`/`,` |
| `report()` | Journal complet des opérations |
